# 1.4 低秩分解（Low-Rank Factorization）

## 什么是低秩分解？

低秩分解将大型权重矩阵（或张量）分解为两个或多个低秩因子的乘积，减少参数量和计算量。核心思想是：大多数权重矩阵的有效秩远小于其名义维度，存在大量冗余。

## 为什么用低秩分解？

1. **参数冗余**：神经网络权重矩阵通常是低秩的或近似低秩的，SVD分解后大部分奇异值接近0，截断它们对精度影响极小。
2. **计算量降低**：原始矩阵乘法$O(mn)$分解为两次小矩阵乘法$O(r(m+n))$，当$r \ll \min(m,n)$时计算量大幅减少。
3. **与LoRA的关联**：LoRA本质上就是一种低秩分解——将权重增量$\Delta W$分解为$AB$，仅训练低秩部分。
4. **端侧友好**：分解后的小矩阵更适合端侧设备的有限缓存和计算能力。

## 低秩分解的数学原理

对于权重矩阵$W \in \mathbb{R}^{m \times n}$，假设其有效秩为$r$（$r \ll \min(m,n)$），则：

$$W \approx AB, \quad A \in \mathbb{R}^{m \times r}, B \in \mathbb{R}^{r \times n}$$

参数量从$mn$降至$r(m+n)$，压缩比为$\frac{mn}{r(m+n)}$。当$r \ll \min(m,n)$时，压缩比接近$\frac{\min(m,n)}{r}$。

其中：
- $W$：原始权重矩阵
- $A$：左因子矩阵
- $B$：右因子矩阵
- $r$：分解的秩，控制压缩比与精度的权衡

## 低秩分解方法概览

| 方法 | 适用对象 | 分解形式 | 推理开销 | 是否需微调 | 典型场景 |
|------|---------|---------|---------|-----------|----------|
| 截断SVD | 2D矩阵 | $W \approx AB$ | 两次小矩阵乘 | 可选 | 线性层压缩 |
| Tucker分解 | N维张量 | $\mathcal{X} \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \cdots$ | 核张量+因子矩阵 | 可选 | 卷积核压缩 |
| CP分解 | N维张量 | $\mathcal{X} \approx \sum_{i=1}^r \lambda_i a_i \circ b_i \circ c_i$ | rank-1项求和 | 可选 | 结构化压缩 |
| LoRA | 2D矩阵 | $\Delta W = BA$ | 合并后零开销 | 必须 | 微调+压缩 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import time
from typing import Optional, Tuple, List, Dict

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---
## 1.4.1 SVD分解（Singular Value Decomposition）

### 什么是SVD分解？

对权重矩阵进行奇异值分解 $W = U\Sigma V^T$，保留前r个最大奇异值，得到 $W \approx U_r \Sigma_r V_r^T$，参数量从 $mn$ 降至 $r(m+n)$。

这是最经典的低秩分解方法，通过截断小奇异值来压缩矩阵。SVD提供了在Frobenius范数下的最优低秩近似（Eckart-Young定理）。

### 为什么SVD分解有效？

1. **最优低秩近似**：Eckart-Young定理保证截断SVD给出Frobenius范数下的最优秩r近似。
2. **能量集中**：奇异值按大小排列，前几个奇异值通常包含大部分"能量"（信息），截断小奇异值损失很小。
3. **无需训练**：SVD是纯数学分解，不需要训练数据或反向传播。

### SVD分解的数学公式

**完整SVD**：
$$W = U \Sigma V^T = \sum_{i=1}^{\min(m,n)} \sigma_i u_i v_i^T$$

**截断SVD（保留前r个奇异值）**：
$$W \approx W_r = U_r \Sigma_r V_r^T = \sum_{i=1}^{r} \sigma_i u_i v_i^T$$

**分解为两个矩阵**：
$$A = U_r \Sigma_r \in \mathbb{R}^{m \times r}, \quad B = V_r^T \in \mathbb{R}^{r \times n}$$

**能量保留比**：
$$\text{Energy}(r) = \frac{\sum_{i=1}^{r} \sigma_i^2}{\sum_{i=1}^{\min(m,n)} \sigma_i^2}$$

其中：
- $W \in \mathbb{R}^{m \times n}$：原始权重矩阵
- $U \in \mathbb{R}^{m \times m}$：左奇异向量矩阵
- $\Sigma \in \mathbb{R}^{m \times n}$：奇异值对角矩阵，$\sigma_1 \geq \sigma_2 \geq ... \geq 0$
- $V \in \mathbb{R}^{n \times n}$：右奇异向量矩阵
- $r$：保留的奇异值数量（秩）
- $\sigma_i$：第$i$个奇异值，越大表示该方向越重要
- 能量保留比通常取90%-99%来确定合适的秩$r$

### SVD秩选择策略

SVD秩$r$的选择是压缩比与精度权衡的核心，常用方法如下：

1. **能量保留法（推荐）**：设定目标能量保留比，自动确定秩
   - **90%能量**：激进压缩，参数减少最多，但精度损失较大。适合对精度要求不高的场景
   - **95%能量**：平衡选择，压缩比与精度兼顾。最常用的默认值
   - **99%能量**：保守压缩，几乎不损失精度，但压缩比有限。适合精度敏感场景
   - 实践建议：从95%开始，在验证集上评估精度，根据需要调整

2. **固定比例法**：取$\min(m,n)$的一定比例作为秩
   - 通常取25%-50%
   - 简单但不够精确，不同层的最优比例可能差异很大

3. **敏感性分析法**：逐层搜索最优秩
   - 对每层尝试不同秩，在验证集上评估精度
   - 精度最高但成本最大，适合最终部署前的精细调优

4. **实践经验**：
   - FFN中间层（维度大）通常可以更激进压缩（50%+）
   - 注意力投影层（QKV）对压缩更敏感，建议保守（75%+）
   - 首尾层（embedding和分类头）通常不压缩

In [ ]:
def svd_compress(weight: torch.Tensor, rank: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """SVD低秩压缩

    Args:
        weight: 原始权重矩阵 [out_features, in_features]
        rank: 保留的秩

    Returns:
        A: 左因子矩阵 [out_features, rank]
        B: 右因子矩阵 [rank, in_features]
        S: 完整奇异值（用于分析）
    """
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)
    U_r = U[:, :rank]
    S_r = S[:rank]
    Vh_r = Vh[:rank, :]
    A = U_r @ torch.diag(S_r)
    B = Vh_r
    return A, B, S


def svd_decompress(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """SVD低秩重建"""
    return A @ B


def compute_energy_ratio(S: torch.Tensor, rank: int) -> float:
    """计算前rank个奇异值的能量保留比"""
    total_energy = (S ** 2).sum()
    retained_energy = (S[:rank] ** 2).sum()
    return (retained_energy / total_energy).item()


def find_rank_for_energy(S: torch.Tensor, target_energy: float) -> int:
    """根据目标能量保留比自动确定秩"""
    total_energy = (S ** 2).sum()
    cumulative = torch.cumsum(S ** 2, dim=0) / total_energy
    rank = (cumulative < target_energy).sum().item() + 1
    return min(rank, len(S))


m, n = 256, 512
rank_true = 32
U_true = torch.randn(m, rank_true)
V_true = torch.randn(rank_true, n)
W_lowrank = U_true @ V_true + torch.randn(m, n) * 0.01
W_full = torch.randn(m, n)

print(f"=== SVD低秩压缩 ===")
print(f"原始矩阵: {m}x{n} = {m*n} 参数")

for W, name in [(W_lowrank, "低秩矩阵(有效秩≈32)"), (W_full, "稠密随机矩阵")]:
    print(f"\n--- {name} ---")
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    total_energy = (S ** 2).sum()
    cumulative = torch.cumsum(S ** 2, dim=0) / total_energy
    for threshold in [0.9, 0.95, 0.99]:
        r = find_rank_for_energy(S, threshold)
        r = min(r, min(m, n))
        A, B, _ = svd_compress(W, rank=r)
        W_recon = svd_decompress(A, B)
        mse = F.mse_loss(W, W_recon)
        param_ratio = (m * r + r * n) / (m * n)
        print(f"  {threshold:.0%}能量保留: rank={r}, 参数比={param_ratio:.2%}, MSE={mse:.6f}")

ranks = [8, 16, 32, 64, 128]
print(f"\n--- 低秩矩阵不同rank的压缩效果 ---")
for r in ranks:
    A, B, _ = svd_compress(W_lowrank, rank=r)
    W_recon = svd_decompress(A, B)
    mse = F.mse_loss(W_lowrank, W_recon)
    cos = F.cosine_similarity(W_lowrank.unsqueeze(0), W_recon.unsqueeze(0)).mean()
    param_ratio = (m * r + r * n) / (m * n)
    energy = compute_energy_ratio(torch.linalg.svd(W_lowrank, full_matrices=False)[1], r)
    print(f"  rank={r:<4} 参数比={param_ratio:.2%} MSE={mse:.6f} 余弦相似度={cos:.6f} 能量保留={energy:.2%}")

---
### SVD压缩在Transformer线性层中的应用

#### 什么是SVD压缩在Transformer中的应用？

将SVD分解应用于Transformer中的线性层（QKV投影、输出投影、FFN层），用两个小矩阵替代原始大矩阵，实现模型压缩。

#### 为什么对Transformer线性层做SVD压缩？

1. **参数集中**：Transformer的大部分参数在线性层（约70%在FFN层），压缩线性层收益最大
2. **低秩特性**：FFN层的中间维度通常远大于隐藏维度，存在大量冗余
3. **精度可控**：通过选择保留的奇异值数量，精确控制压缩比与精度权衡

#### 数学原理

对于线性层$y = Wx + b$，$W \in \mathbb{R}^{m \times n}$：
$$W \approx U_r \Sigma_r V_r^T = A \cdot B$$
其中$A = U_r \Sigma_r \in \mathbb{R}^{m \times r}$，$B = V_r^T \in \mathbb{R}^{r \times n}$

推理时：$y = ABx + b$，两次小矩阵乘法替代一次大矩阵乘法
- 原始计算量：$O(mn)$
- 压缩后计算量：$O(r(m+n))$
- 压缩比：$\frac{mn}{r(m+n)}$

In [ ]:
class SVDCompressedLinear(nn.Module):
    """SVD压缩的线性层

    将原始线性层 W = AB 分解为两个低秩矩阵，推理时执行两次小矩阵乘法。
    实际应用中，通常对FFN层和大型投影层进行SVD压缩，保留注意力层不变。
    """

    def __init__(self, A: torch.Tensor, B: torch.Tensor, bias: Optional[torch.Tensor] = None):
        super().__init__()
        self.A = nn.Parameter(A)
        self.B = nn.Parameter(B)
        if bias is not None:
            self.bias = nn.Parameter(bias)
        else:
            self.register_parameter('bias', None)
        self.rank = A.shape[1]
        self.in_features = B.shape[1]
        self.out_features = A.shape[0]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x @ self.B.t() @ self.A.t()
        if self.bias is not None:
            out = out + self.bias
        return out

    @classmethod
    def from_linear(cls, linear: nn.Linear, rank: int) -> 'SVDCompressedLinear':
        """从标准线性层创建SVD压缩层

        Args:
            linear: 原始nn.Linear层
            rank: 压缩后的秩
        """
        W = linear.weight.data
        A, B, _ = svd_compress(W, rank=rank)
        bias = linear.bias.data.clone() if linear.bias is not None else None
        return cls(A, B, bias)

    def extra_repr(self) -> str:
        return (f'in_features={self.in_features}, out_features={self.out_features}, '
                f'rank={self.rank}, bias={self.bias is not None}')


def replace_linear_with_svd(module: nn.Module, rank_ratio: float = 0.5,
                             min_rank: int = 4, skip_layers: Optional[List[str]] = None) -> nn.Module:
    """递归替换模型中的Linear层为SVD压缩版本

    Args:
        module: 原始模型
        rank_ratio: 秩占min(out,in)的比例
        min_rank: 最小秩，避免过度压缩
        skip_layers: 跳过不压缩的层名（支持子串匹配）

    Returns:
        压缩后的模型（原地修改）
    """
    if skip_layers is None:
        skip_layers = []

    for name, child in module.named_children():
        full_name = name
        if isinstance(child, nn.Linear):
            should_skip = any(s in full_name for s in skip_layers)
            if should_skip:
                continue
            out_f, in_f = child.weight.shape
            rank = max(min_rank, int(min(out_f, in_f) * rank_ratio))
            compressed = SVDCompressedLinear.from_linear(child, rank=rank)
            setattr(module, name, compressed)
        else:
            replace_linear_with_svd(child, rank_ratio, min_rank, skip_layers)

    return module


class SmallTransformer(nn.Module):
    def __init__(self, dim: int = 64, n_heads: int = 4, n_classes: int = 10):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return self.head(x.mean(dim=1))


dim = 64
model = SmallTransformer(dim=dim, n_heads=4, n_classes=10).to(device)
original_params = sum(p.numel() for p in model.parameters())

x_test = torch.randn(32, 8, dim, device=device)
y_test = torch.randint(0, 10, (32,))

with torch.no_grad():
    out_orig = model(x_test)
    acc_orig = (out_orig.argmax(1).cpu() == y_test).float().mean()

print(f"=== SVD压缩Transformer线性层 ===")
print(f"原始参数量: {original_params:,}")
print(f"\n--- 仅压缩FFN层（跳过注意力层和分类头） ---")

for rank_ratio in [0.25, 0.5, 0.75]:
    model_copy = copy.deepcopy(model)
    replace_linear_with_svd(
        model_copy,
        rank_ratio=rank_ratio,
        min_rank=4,
        skip_layers=['attn', 'head']
    )

    compressed_params = sum(p.numel() for p in model_copy.parameters())
    with torch.no_grad():
        out_comp = model_copy(x_test)
        acc_comp = (out_comp.argmax(1).cpu() == y_test).float().mean()
        output_diff = (out_orig - out_comp).norm() / out_orig.norm() * 100

    print(f"  rank比例={rank_ratio:.0%}: 参数量={compressed_params:,}({compressed_params/original_params:.1%}), "
          f"输出差异={output_diff:.2f}%, 准确率={acc_comp:.4f}")

print(f"\n--- 压缩所有线性层 ---")
for rank_ratio in [0.25, 0.5, 0.75]:
    model_copy = copy.deepcopy(model)
    replace_linear_with_svd(model_copy, rank_ratio=rank_ratio, min_rank=4)

    compressed_params = sum(p.numel() for p in model_copy.parameters())
    with torch.no_grad():
        out_comp = model_copy(x_test)
        acc_comp = (out_comp.argmax(1).cpu() == y_test).float().mean()
        output_diff = (out_orig - out_comp).norm() / out_orig.norm() * 100

    print(f"  rank比例={rank_ratio:.0%}: 参数量={compressed_params:,}({compressed_params/original_params:.1%}), "
          f"输出差异={output_diff:.2f}%, 准确率={acc_comp:.4f}")

---
## 1.4.2 Tucker分解

### 什么是Tucker分解？

Tucker分解是对高维张量进行多模式分解的方法，将张量分解为一个核心张量（core tensor）和各模式的因子矩阵（factor matrices）。它是SVD在高维张量上的推广，适合多维权重张量（如卷积核4D张量）的压缩。

对于2D矩阵，Tucker-2分解等价于截断SVD；对于4D卷积核，Tucker分解可以同时对输出通道、输入通道、空间维度进行压缩，这是SVD无法做到的。

### 为什么用Tucker分解？

1. **高维张量压缩**：卷积核是4D张量（输出通道×输入通道×高×宽），SVD只能处理2D矩阵，Tucker可以处理任意维度的张量。
2. **各维度独立压缩**：Tucker分解允许每个维度选择不同的秩，比SVD更灵活。例如卷积核的空间维度（3×3）通常不需要压缩，而通道维度可以大幅压缩。
3. **保留空间结构**：对卷积核的空间维度单独处理，可以保留空间局部性。
4. **与卷积运算兼容**：Tucker分解后的卷积核可以等价地转换为多个小卷积的级联，无需修改推理框架。

### Tucker分解的数学公式

Tucker分解将张量 $\mathcal{X} \in \mathbb{R}^{I_1 \times I_2 \times \cdots \times I_N}$ 分解为核心张量 $\mathcal{G}$ 和各模式的因子矩阵：
$$\mathcal{X} \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \cdots \times_N U^{(N)}$$

其中：
- $\mathcal{X}$：原始N维张量
- $\mathcal{G} \in \mathbb{R}^{R_1 \times R_2 \times \cdots \times R_N}$：核心张量，$R_i \leq I_i$
- $U^{(i)} \in \mathbb{R}^{I_i \times R_i}$：第$i$个模式的因子矩阵
- $\times_i$：第$i$个模式的张量-矩阵乘法（mode-i product）
- $R_i$：第$i$个模式的Tucker秩，控制该维度的压缩程度
- 参数量从$\prod I_i$降至$\prod R_i + \sum I_i R_i$

### Mode-i Product（模式乘法）

Mode-i product是Tucker分解的核心运算，表示张量沿第$i$个模式与矩阵相乘：
$$(\mathcal{X} \times_i U)_{j_1, \ldots, j_{i-1}, k, j_{i+1}, \ldots, j_N} = \sum_{j_i=1}^{I_i} \mathcal{X}_{j_1, \ldots, j_N} \cdot U_{j_i, k}$$

直观理解：mode-i product等价于将张量的第$i$个维度"投影"到更低维空间。

### Tucker-2分解（卷积核最常用）

对于4D卷积核$W \in \mathbb{R}^{C_{out} \times C_{in} \times H \times W}$，Tucker-2分解仅对前两个模式（通道维度）分解，保留空间维度不变：
$$W \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)}$$

其中：
- $\mathcal{G} \in \mathbb{R}^{R_{out} \times R_{in} \times H \times W}$：核心张量
- $U^{(1)} \in \mathbb{R}^{C_{out} \times R_{out}}$：输出通道因子矩阵
- $U^{(2)} \in \mathbb{R}^{C_{in} \times R_{in}}$：输入通道因子矩阵
- 参数量从$C_{out} \cdot C_{in} \cdot H \cdot W$降至$R_{out} \cdot R_{in} \cdot H \cdot W + C_{out} \cdot R_{out} + C_{in} \cdot R_{in}$

### HOSVD算法（Higher-Order SVD）

HOSVD是计算Tucker分解的经典算法，通过对每个模式展开后的矩阵做SVD来获取因子矩阵：

1. **模式展开**：将张量沿第$i$个模式展开为矩阵$X_{(i)} \in \mathbb{R}^{I_i \times \prod_{j \neq i} I_j}$
2. **SVD分解**：对每个$X_{(i)}$做截断SVD，取前$R_i$个左奇异向量作为$U^{(i)}$
3. **计算核心张量**：$\mathcal{G} = \mathcal{X} \times_1 {U^{(1)}}^T \times_2 {U^{(2)}}^T \cdots \times_N {U^{(N)}}^T$

HOSVD不是最优的Tucker分解（最优需要交替最小二乘ALS迭代），但计算简单高效，实际效果接近最优。

In [ ]:
def mode_n_product(tensor: torch.Tensor, matrix: torch.Tensor, mode: int) -> torch.Tensor:
    """张量的mode-n乘法

    Args:
        tensor: 输入张量 [..., I_n, ...]
        matrix: 矩阵 [R, I_n] 或 [I_n, R]
        mode: 模式索引（0-based）

    Returns:
        结果张量
    """
    ndim = tensor.dim()
    perm = list(range(ndim))
    perm[0], perm[mode] = perm[mode], perm[0]
    tensor_t = tensor.permute(perm)
    shape = tensor_t.shape
    tensor_mat = tensor_t.reshape(shape[0], -1)
    result_mat = matrix @ tensor_mat
    new_shape = list(shape)
    new_shape[0] = result_mat.shape[0]
    result = result_mat.reshape(new_shape)
    inv_perm = [0] * ndim
    for i, p in enumerate(perm):
        inv_perm[p] = i
    return result.permute(inv_perm)


def unfold_tensor(tensor: torch.Tensor, mode: int) -> torch.Tensor:
    """将张量沿第mode个模式展开为矩阵

    Args:
        tensor: N维张量
        mode: 模式索引（0-based）

    Returns:
        展开矩阵 [I_mode, prod(other_dims)]
    """
    ndim = tensor.dim()
    perm = list(range(ndim))
    perm.pop(mode)
    perm = [mode] + perm
    tensor_t = tensor.permute(perm)
    return tensor_t.reshape(tensor.shape[mode], -1)


def hosvd(tensor: torch.Tensor, ranks: List[int]) -> Tuple[torch.Tensor, List[torch.Tensor]]:
    """Higher-Order SVD (HOSVD) 计算Tucker分解

    通过对每个模式展开矩阵做SVD来获取因子矩阵，然后计算核心张量。

    Args:
        tensor: 原始N维张量
        ranks: 各模式的目标秩 [R_0, R_1, ..., R_{N-1}]

    Returns:
        core: 核心张量 [R_0, R_1, ..., R_{N-1}]
        factors: 各模式因子矩阵列表，factors[i] 形状为 [I_i, R_i]
    """
    ndim = tensor.dim()
    assert len(ranks) == ndim, f"ranks长度({len(ranks)})必须等于张量维度({ndim})"

    factors = []
    for mode in range(ndim):
        unfolded = unfold_tensor(tensor, mode)
        U, S, Vh = torch.linalg.svd(unfolded, full_matrices=False)
        factors.append(U[:, :ranks[mode]])

    core = tensor.clone()
    for mode in range(ndim):
        core = mode_n_product(core, factors[mode].t(), mode)

    return core, factors


def tucker_reconstruct(core: torch.Tensor, factors: List[torch.Tensor]) -> torch.Tensor:
    """从Tucker分解结果重建原始张量

    Args:
        core: 核心张量
        factors: 各模式因子矩阵列表

    Returns:
        重建张量
    """
    result = core
    for mode, factor in enumerate(factors):
        result = mode_n_product(result, factor, mode)
    return result


def find_tucker_ranks(tensor: torch.Tensor, energy_threshold: float = 0.9) -> List[int]:
    """根据能量保留比自动确定各模式的Tucker秩

    Args:
        tensor: 原始张量
        energy_threshold: 目标能量保留比

    Returns:
        各模式的秩列表
    """
    ranks = []
    for mode in range(tensor.dim()):
        unfolded = unfold_tensor(tensor, mode)
        S = torch.linalg.svdvals(unfolded)
        rank = find_rank_for_energy(S, energy_threshold)
        ranks.append(rank)
    return ranks


print("=== Tucker分解基础验证 ===")

W_2d = torch.randn(256, 512)
print(f"\n--- 2D矩阵 Tucker分解 vs SVD ---")
for ratio in [0.25, 0.5, 0.75]:
    m, n = W_2d.shape
    rank_m = max(1, int(m * ratio))
    rank_n = max(1, int(n * ratio))
    core, factors = hosvd(W_2d, ranks=[rank_m, rank_n])
    W_recon = tucker_reconstruct(core, factors)
    mse = F.mse_loss(W_2d, W_recon)
    orig_params = W_2d.numel()
    comp_params = core.numel() + sum(f.numel() for f in factors)
    print(f"  ranks=[{rank_m},{rank_n}] 参数比={comp_params/orig_params:.2%}, MSE={mse:.6f}")

W_3d = torch.randn(64, 128, 32)
print(f"\n--- 3D张量 Tucker分解 ---")
print(f"原始张量形状: {list(W_3d.shape)}, 参数量: {W_3d.numel():,}")
auto_ranks = find_tucker_ranks(W_3d, energy_threshold=0.95)
print(f"自动确定秩(95%能量): {auto_ranks}")
for ranks in [[16, 32, 16], [32, 64, 16], [32, 64, 32]]:
    core, factors = hosvd(W_3d, ranks=ranks)
    W_recon = tucker_reconstruct(core, factors)
    mse = F.mse_loss(W_3d, W_recon)
    orig_params = W_3d.numel()
    comp_params = core.numel() + sum(f.numel() for f in factors)
    print(f"  ranks={ranks} 参数比={comp_params/orig_params:.2%}, MSE={mse:.6f}")

---
### Tucker分解压缩卷积层

#### 卷积核的Tucker-2分解

对于4D卷积核$W \in \mathbb{R}^{C_{out} \times C_{in} \times K_H \times K_W}$，Tucker-2分解仅对通道维度分解：
$$W \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)}$$

这等价于将原始卷积分解为三个级联卷积：
1. **1×1卷积**（输入通道压缩）：$C_{in} \to R_{in}$，权重为${U^{(2)}}^T$
2. **空间卷积**（核心卷积）：$R_{in} \to R_{out}$，核大小$K_H \times K_W$，权重为$\mathcal{G}$
3. **1×1卷积**（输出通道恢复）：$R_{out} \to C_{out}$，权重为$U^{(1)}$

#### 计算量分析

| 运算 | 原始卷积 | Tucker分解后 |
|------|---------|-------------|
| FLOPs | $C_{out} \cdot C_{in} \cdot K_H \cdot K_W \cdot H_{out} \cdot W_{out}$ | $(C_{in} \cdot R_{in} + R_{in} \cdot R_{out} \cdot K_H \cdot K_W + R_{out} \cdot C_{out}) \cdot H_{out} \cdot W_{out}$ |
| 参数量 | $C_{out} \cdot C_{in} \cdot K_H \cdot K_W$ | $C_{in} \cdot R_{in} + R_{in} \cdot R_{out} \cdot K_H \cdot K_W + R_{out} \cdot C_{out}$ |

当$R_{in} \ll C_{in}$且$R_{out} \ll C_{out}$时，计算量和参数量都大幅减少。

In [ ]:
class TuckerCompressedConv2d(nn.Module):
    """Tucker分解压缩的Conv2d层

    将原始卷积分解为三个级联卷积：
    1. 1x1卷积: C_in -> R_in (输入通道压缩)
    2. 空间卷积: R_in -> R_out, kernel_size=原始kernel_size (核心卷积)
    3. 1x1卷积: R_out -> C_out (输出通道恢复)

    实际应用中，通常对中间层的大卷积进行Tucker压缩，
    保留首尾层和深度可分离卷积不变。
    """

    def __init__(
        self,
        conv1x1_in: nn.Conv2d,
        conv_spatial: nn.Conv2d,
        conv1x1_out: nn.Conv2d,
        original_in_channels: int,
        original_out_channels: int,
    ):
        super().__init__()
        self.conv1x1_in = conv1x1_in
        self.conv_spatial = conv_spatial
        self.conv1x1_out = conv1x1_out
        self.original_in_channels = original_in_channels
        self.original_out_channels = original_out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1x1_in(x)
        x = self.conv_spatial(x)
        x = self.conv1x1_out(x)
        return x

    @classmethod
    def from_conv2d(
        cls,
        conv: nn.Conv2d,
        rank_in: int,
        rank_out: int,
    ) -> 'TuckerCompressedConv2d':
        """从标准Conv2d层创建Tucker压缩版本

        Args:
            conv: 原始nn.Conv2d层
            rank_in: 输入通道压缩后的秩
            rank_out: 输出通道压缩后的秩
        """
        W = conv.weight.data
        C_out, C_in, K_H, K_W = W.shape
        stride = conv.stride
        padding = conv.padding

        W_unfold0 = W.reshape(C_out, C_in * K_H * K_W)
        U_out_full, _, _ = torch.linalg.svd(W_unfold0, full_matrices=False)
        U_out = U_out_full[:, :rank_out]

        W_unfold1 = W.permute(1, 0, 2, 3).reshape(C_in, C_out * K_H * K_W)
        U_in_full, _, _ = torch.linalg.svd(W_unfold1, full_matrices=False)
        U_in = U_in_full[:, :rank_in]

        core = mode_n_product(W, U_out.t(), 0)
        core = mode_n_product(core, U_in.t(), 1)

        conv1x1_in = nn.Conv2d(
            C_in, rank_in, kernel_size=1, bias=False,
            stride=1, padding=0
        )
        conv1x1_in.weight.data = U_in.t().unsqueeze(-1).unsqueeze(-1)

        conv_spatial = nn.Conv2d(
            rank_in, rank_out, kernel_size=(K_H, K_W),
            stride=stride, padding=padding, bias=False
        )
        conv_spatial.weight.data = core

        conv1x1_out = nn.Conv2d(
            rank_out, C_out, kernel_size=1, bias=True,
            stride=1, padding=0
        )
        conv1x1_out.weight.data = U_out.unsqueeze(-1).unsqueeze(-1)
        if conv.bias is not None:
            conv1x1_out.bias.data = conv.bias.data.clone()
        else:
            conv1x1_out.bias.data.zero_()

        return cls(conv1x1_in, conv_spatial, conv1x1_out, C_in, C_out)

    def extra_repr(self) -> str:
        r_in = self.conv1x1_in.out_channels
        r_out = self.conv1x1_out.in_channels
        return (f'original: ({self.original_in_channels} -> {self.original_out_channels}), '
                f'compressed: ({self.original_in_channels} -> {r_in} -> {r_out} -> {self.original_out_channels})')


def compute_conv_flops(conv: nn.Conv2d, input_h: int, input_w: int) -> int:
    """计算Conv2d层的FLOPs"""
    C_out, C_in, K_H, K_W = conv.weight.shape
    out_h = (input_h + 2 * conv.padding[0] - K_H) // conv.stride[0] + 1
    out_w = (input_w + 2 * conv.padding[1] - K_W) // conv.stride[1] + 1
    return 2 * C_out * C_in * K_H * K_W * out_h * out_w


def compute_tucker_conv_flops(tucker_conv: TuckerCompressedConv2d, input_h: int, input_w: int) -> int:
    """计算Tucker压缩Conv2d的FLOPs"""
    flops = 0
    h, w = input_h, input_w
    for conv in [tucker_conv.conv1x1_in, tucker_conv.conv_spatial, tucker_conv.conv1x1_out]:
        C_out, C_in_k, K_H, K_W = conv.weight.shape
        out_h = (h + 2 * conv.padding[0] - K_H) // conv.stride[0] + 1
        out_w = (w + 2 * conv.padding[1] - K_W) // conv.stride[1] + 1
        flops += 2 * C_out * C_in_k * K_H * K_W * out_h * out_w
        h, w = out_h, out_w
    return flops


C_out, C_in, K = 128, 256, 3
conv_orig = nn.Conv2d(C_in, C_out, kernel_size=K, padding=1, bias=True)

print(f"=== Tucker分解压缩Conv2d ===")
print(f"原始卷积: Conv2d({C_in}, {C_out}, kernel_size={K})")
print(f"原始参数量: {sum(p.numel() for p in conv_orig.parameters()):,}")

input_h, input_w = 32, 32
orig_flops = compute_conv_flops(conv_orig, input_h, input_w)
print(f"原始FLOPs: {orig_flops:,}")

x_conv = torch.randn(1, C_in, input_h, input_w)
with torch.no_grad():
    out_orig = conv_orig(x_conv)

print(f"\n--- 不同秩的压缩效果 ---")
for r_in, r_out in [(64, 32), (128, 64), (128, 128), (192, 96)]:
    tucker_conv = TuckerCompressedConv2d.from_conv2d(conv_orig, rank_in=r_in, rank_out=r_out)
    comp_params = sum(p.numel() for p in tucker_conv.parameters())
    comp_flops = compute_tucker_conv_flops(tucker_conv, input_h, input_w)

    with torch.no_grad():
        out_comp = tucker_conv(x_conv)
    output_diff = (out_orig - out_comp).norm() / out_orig.norm() * 100

    print(f"  ranks=({r_in}, {r_out}): 参数={comp_params:,}({comp_params/conv_orig.weight.numel():.1%}), "
          f"FLOPs={comp_flops:,}({comp_flops/orig_flops:.1%}), 输出差异={output_diff:.2f}%")

---
### Tucker分解压缩CNN模型

#### 实际应用中的Tucker压缩策略

1. **选择性压缩**：只压缩参数量大的中间层卷积，保留首尾层（输入层和分类头）
2. **秩选择策略**：
   - 基于能量保留比自动确定（推荐95%以上）
   - 基于经验比例（通常取原始通道数的25%-50%）
   - 基于敏感性分析（逐层搜索最优秩）
3. **微调恢复**：压缩后通常需要少量微调来恢复精度损失

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Linear(128 * 4 * 4, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


def replace_conv_with_tucker(
    module: nn.Module,
    rank_ratio: float = 0.5,
    min_rank: int = 4,
    skip_layers: Optional[List[str]] = None,
) -> Dict[str, dict]:
    """递归替换模型中的Conv2d层为Tucker压缩版本

    Args:
        module: 原始模型
        rank_ratio: 秩占原始通道数的比例
        min_rank: 最小秩
        skip_layers: 跳过不压缩的层名

    Returns:
        压缩信息字典 {层名: {原始参数, 压缩参数, rank_in, rank_out}}
    """
    if skip_layers is None:
        skip_layers = []

    compression_info = {}

    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d) and child.weight.dim() == 4:
            should_skip = any(s in name for s in skip_layers)
            if should_skip:
                continue
            C_out, C_in = child.weight.shape[:2]
            if C_out < 2 * min_rank or C_in < 2 * min_rank:
                continue

            rank_in = max(min_rank, int(C_in * rank_ratio))
            rank_out = max(min_rank, int(C_out * rank_ratio))
            orig_params = sum(p.numel() for p in child.parameters())

            tucker_conv = TuckerCompressedConv2d.from_conv2d(child, rank_in=rank_in, rank_out=rank_out)
            comp_params = sum(p.numel() for p in tucker_conv.parameters())

            setattr(module, name, tucker_conv)
            compression_info[name] = {
                'orig_params': orig_params,
                'comp_params': comp_params,
                'rank_in': rank_in,
                'rank_out': rank_out,
                'C_in': C_in,
                'C_out': C_out,
            }
        else:
            child_info = replace_conv_with_tucker(child, rank_ratio, min_rank, skip_layers)
            compression_info.update(child_info)

    return compression_info


cnn_model = SmallCNN(num_classes=10).to(device)
original_params = sum(p.numel() for p in cnn_model.parameters())

x_cnn = torch.randn(8, 3, 32, 32, device=device)
with torch.no_grad():
    out_cnn_orig = cnn_model(x_cnn)

print(f"=== Tucker分解压缩CNN ===")
print(f"原始模型参数量: {original_params:,}")

for ratio in [0.25, 0.5, 0.75]:
    model_copy = copy.deepcopy(cnn_model)
    info = replace_conv_with_tucker(
        model_copy,
        rank_ratio=ratio,
        min_rank=4,
        skip_layers=['0']
    )

    comp_params = sum(p.numel() for p in model_copy.parameters())
    with torch.no_grad():
        out_comp = model_copy(x_cnn)
    output_diff = (out_cnn_orig - out_comp).norm() / out_cnn_orig.norm() * 100

    print(f"\n  rank比例={ratio:.0%}: 总参数={comp_params:,}({comp_params/original_params:.1%}), 输出差异={output_diff:.2f}%")
    for layer_name, layer_info in info.items():
        print(f"    {layer_name}: ({layer_info['C_in']},{layer_info['C_out']}) -> "
              f"ranks=({layer_info['rank_in']},{layer_info['rank_out']}), "
              f"参数 {layer_info['orig_params']}->{layer_info['comp_params']} "
              f"({layer_info['comp_params']/layer_info['orig_params']:.1%})")

---
### Tucker分解秩选择策略

#### 如何选择合适的Tucker秩？

Tucker秩的选择直接影响压缩比与精度的权衡。以下是几种常用的秩选择策略：

1. **能量保留法（推荐）**：对每个模式展开矩阵做SVD，保留累计能量达到阈值的奇异值数量
   - **95%能量**：平衡选择，最常用的默认值
   - **90%能量**：激进压缩，适合参数量要求严格的场景
   - **99%能量**：保守压缩，精度损失极小
   - 各模式可以使用不同的能量阈值（如通道维度95%，空间维度99%）
2. **固定比例法**：取原始维度的一定比例（如25%-50%），简单但可能不够精确
   - 适合快速原型验证，不推荐最终部署
3. **VBD（Variational Bayesian）法**：使用贝叶斯方法自动确定各模式的最优秩
   - 理论最优但实现复杂，适合研究场景
4. **敏感性分析法**：逐层搜索最优秩，在验证集上评估精度损失
   - 精度最高但成本最大，适合最终部署前的精细调优

#### Tucker-2分解的秩选择实践

对于卷积核的Tucker-2分解（仅压缩通道维度），秩选择有额外考虑：

- **$R_{in}$（输入通道秩）**：通常取$C_{in}$的25%-50%，因为输入通道冗余度通常较高
- **$R_{out}$（输出通道秩）**：通常取$C_{out}$的25%-50%，但输出通道冗余度可能较低
- **非对称压缩**：$R_{in}$和$R_{out}$可以不同。经验上$R_{in} > R_{out}$通常效果更好（先压缩输入通道更多）
- **空间维度**：通常不压缩（保持$K_H \times K_W$不变），因为3×3等小核的空间信息很重要
- **首尾层**：第一个卷积层（3通道输入）和最后一个卷积层通常不压缩

In [ ]:
print("=== Tucker秩选择策略对比 ===")

C_out, C_in, K = 256, 512, 3
conv_layer = nn.Conv2d(C_in, C_out, kernel_size=K, padding=1)
W = conv_layer.weight.data

print(f"卷积核形状: [{C_out}, {C_in}, {K}, {K}], 参数量: {W.numel():,}")

def tucker2_decompose(W: torch.Tensor, rank_out: int, rank_in: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Tucker-2分解：仅对通道维度分解，保留空间维度不变"""
    C_out, C_in, K_H, K_W = W.shape
    W_unfold0 = W.reshape(C_out, C_in * K_H * K_W)
    U_out, _, _ = torch.linalg.svd(W_unfold0, full_matrices=False)
    U_out = U_out[:, :rank_out]
    W_unfold1 = W.permute(1, 0, 2, 3).reshape(C_in, C_out * K_H * K_W)
    U_in, _, _ = torch.linalg.svd(W_unfold1, full_matrices=False)
    U_in = U_in[:, :rank_in]
    core = mode_n_product(W, U_out.t(), 0)
    core = mode_n_product(core, U_in.t(), 1)
    return core, U_out, U_in

def tucker2_reconstruct(core: torch.Tensor, U_out: torch.Tensor, U_in: torch.Tensor) -> torch.Tensor:
    """Tucker-2重建"""
    W_recon = mode_n_product(core, U_out, 0)
    W_recon = mode_n_product(W_recon, U_in, 1)
    return W_recon

def tucker2_compressed_params(C_out: int, C_in: int, K_H: int, K_W: int, rank_out: int, rank_in: int) -> int:
    """计算Tucker-2压缩后的参数量"""
    return C_out * rank_out + rank_out * rank_in * K_H * K_W + C_in * rank_in

print(f"\n--- 策略1: 能量保留法（基于通道模式展开的SVD） ---")
W_unfold0 = W.reshape(C_out, C_in * K * K)
S_out = torch.linalg.svdvals(W_unfold0)
W_unfold1 = W.permute(1, 0, 2, 3).reshape(C_in, C_out * K * K)
S_in = torch.linalg.svdvals(W_unfold1)
for threshold in [0.90, 0.95, 0.99]:
    r_out = find_rank_for_energy(S_out, threshold)
    r_in = find_rank_for_energy(S_in, threshold)
    core, U_out, U_in = tucker2_decompose(W, r_out, r_in)
    W_recon = tucker2_reconstruct(core, U_out, U_in)
    mse = F.mse_loss(W, W_recon)
    comp_params = tucker2_compressed_params(C_out, C_in, K, K, r_out, r_in)
    print(f"  {threshold:.0%}能量: ranks=({r_out}, {r_in}), 参数比={comp_params/W.numel():.2%}, MSE={mse:.6f}")

print(f"\n--- 策略2: 固定比例法 ---")
for ratio in [0.25, 0.5, 0.75]:
    r_in = max(4, int(C_in * ratio))
    r_out = max(4, int(C_out * ratio))
    core, U_out, U_in = tucker2_decompose(W, r_out, r_in)
    W_recon = tucker2_reconstruct(core, U_out, U_in)
    mse = F.mse_loss(W, W_recon)
    comp_params = tucker2_compressed_params(C_out, C_in, K, K, r_out, r_in)
    print(f"  {ratio:.0%}比例: ranks=({r_out}, {r_in}), 参数比={comp_params/W.numel():.2%}, MSE={mse:.6f}")

print(f"\n--- 策略3: 非对称压缩（输出通道压缩更多） ---")
for r_in_ratio, r_out_ratio in [(0.5, 0.25), (0.75, 0.25), (0.75, 0.5)]:
    r_in = max(4, int(C_in * r_in_ratio))
    r_out = max(4, int(C_out * r_out_ratio))
    core, U_out, U_in = tucker2_decompose(W, r_out, r_in)
    W_recon = tucker2_reconstruct(core, U_out, U_in)
    mse = F.mse_loss(W, W_recon)
    comp_params = tucker2_compressed_params(C_out, C_in, K, K, r_out, r_in)
    print(f"  in={r_in_ratio:.0%},out={r_out_ratio:.0%}: ranks=({r_out}, {r_in}), "
          f"参数比={comp_params/W.numel():.2%}, MSE={mse:.6f}")

print(f"\n--- 各通道模式奇异值分布分析 ---")
for mode_name, S_vals, dim in [('C_out(模式0)', S_out, C_out), ('C_in(模式1)', S_in, C_in)]:
    total_energy = (S_vals ** 2).sum()
    cumulative = torch.cumsum(S_vals ** 2, dim=0) / total_energy
    r90 = (cumulative < 0.9).sum().item() + 1
    r95 = (cumulative < 0.95).sum().item() + 1
    r99 = (cumulative < 0.99).sum().item() + 1
    print(f"  {mode_name}, dim={dim}: "
          f"90%能量需{r90}维, 95%需{r95}维, 99%需{r99}维")

---
## 1.4.3 CP分解（CANDECOMP/PARAFAC Decomposition）

### 什么是CP分解？

CP分解（CANDECOMP/PARAFAC分解）将张量分解为若干个秩1张量（rank-1 tensors）之和。它是Tucker分解的特殊情况——当核心张量为超对角张量时，Tucker分解退化为CP分解。

对于3D张量 $\mathcal{X} \in \mathbb{R}^{I \times J \times K}$，CP分解为：
$$\mathcal{X} \approx \sum_{r=1}^{R} \lambda_r \cdot a_r \circ b_r \circ c_r$$

其中 $\circ$ 表示外积，$a_r \in \mathbb{R}^I$, $b_r \in \mathbb{R}^J$, $c_r \in \mathbb{R}^K$ 是因子向量，$\lambda_r$ 是权重。

### 为什么用CP分解？

1. **更紧凑的表示**：CP分解没有核心张量，参数量通常比Tucker分解更少。对于3D张量，CP参数量为$R(I+J+K)$，Tucker参数量为$R_1R_2R_3 + IR_1 + JR_2 + KR_3$。
2. **唯一性**：在适当条件下（因子矩阵满足Kruskal条件），CP分解（忽略排列和缩放歧义）是唯一的，而Tucker分解不唯一。
3. **结构化压缩**：每个秩1项可以独立计算，适合并行化。
4. **与卷积的等价性**：CP分解的卷积核可以等价转换为深度可分离卷积的级联，推理效率高。

### CP分解 vs Tucker分解

| 特性 | CP分解 | Tucker分解 |
|------|--------|-----------|
| 分解形式 | 秩1项之和 | 核心张量 + 因子矩阵 |
| 参数量 | $R(I+J+K)$ | $R_1R_2R_3 + IR_1 + JR_2 + KR_3$ |
| 唯一性 | 条件唯一 | 不唯一 |
| 秩选择 | 单一秩R | 各维度独立秩 |
| 表达能力 | 较弱（更紧凑） | 较强（更灵活） |
| 计算方法 | ALS迭代 | HOSVD一次计算 |

### CP分解的数学公式

**元素形式**：
$$x_{ijk} \approx \sum_{r=1}^{R} \lambda_r \cdot a_{ir} \cdot b_{jr} \cdot c_{kr}$$

**矩阵形式（Khatri-Rao积）**：
$$X_{(1)} \approx A \cdot \text{diag}(\lambda) \cdot (C \odot B)^T$$

其中：
- $X_{(1)} \in \mathbb{R}^{I \times JK}$：张量沿模式1的展开矩阵
- $A \in \mathbb{R}^{I \times R}$：模式1的因子矩阵，$A = [a_1, a_2, ..., a_R]$
- $B \in \mathbb{R}^{J \times R}$：模式2的因子矩阵
- $C \in \mathbb{R}^{K \times R}$：模式3的因子矩阵
- $\lambda \in \mathbb{R}^{R}$：权重向量
- $\odot$：Khatri-Rao积（列式Kronecker积）

### CP分解的ALS算法

CP分解通常使用交替最小二乘法（ALS）求解，每次固定其他因子矩阵，优化一个因子矩阵：

1. 固定B、C，求解A：$A = X_{(1)} (C \odot B) (C^T C * B^T B)^{-1}$
2. 固定A、C，求解B：$B = X_{(2)} (C \odot A) (C^T C * A^T A)^{-1}$
3. 固定A、B，求解C：$C = X_{(3)} (B \odot A) (B^T B * A^T A)^{-1}$
4. 归一化：将每列归一化，提取权重$\lambda_r = \|a_r\|$
5. 重复1-4直到收敛

其中 $*$ 表示Hadamard积（逐元素乘法）。

### CP分解的秩选择

CP分解的秩R选择比SVD/Tucker更困难，因为CP分解没有类似SVD的奇异值能量排序：

1. **拟合误差法（推荐）**：逐步增加R，当重构误差下降变缓时停止（肘部法则）。具体做法：
   - 从R=1开始，逐步增加R
   - 计算每个R下的相对重构误差 $\|\mathcal{X} - \hat{\mathcal{X}}\| / \|\mathcal{X}\|$
   - 选择误差下降明显变缓的拐点作为R

2. **核心一致性诊断（CORCONDIA）**：计算CP分解与对应Tucker分解的核心一致性：
   - 值接近100%表示秩合适
   - 低于50%表示秩过大
   - 实现方式：先做CP分解，再用CP的因子矩阵构造Tucker分解，比较核心张量与超对角张量的差异

3. **实践经验**：
   - 对于卷积核压缩，R通常取$\min(C_{out}, C_{in})$的1/4~1/2
   - 起始点可参考Tucker分解的最大秩
   - 在验证集上评估不同R对模型精度的影响
   - CP秩过大会导致过拟合和分解不稳定

In [ ]:
def khatri_rao(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Khatri-Rao积（列式Kronecker积）

    Args:
        A: 矩阵 [I, R]
        B: 矩阵 [J, R]

    Returns:
        Khatri-Rao积 [I*J, R]
    """
    I, R_A = A.shape
    J, R_B = B.shape
    assert R_A == R_B
    return (A.unsqueeze(1) * B.unsqueeze(0)).reshape(I * J, -1)


def cp_als(tensor: torch.Tensor, rank: int, max_iter: int = 200,
           tol: float = 1e-6, verbose: bool = False) -> Tuple[List[torch.Tensor], torch.Tensor, List[float]]:
    """CP分解的ALS算法

    Args:
        tensor: 原始3D张量 [I, J, K]
        rank: CP秩
        max_iter: 最大迭代次数
        tol: 收敛阈值（相对误差变化）
        verbose: 是否打印迭代信息

    Returns:
        factors: 各模式因子矩阵列表 [A, B, C]
        lambdas: 权重向量 [R]
        errors: 每次迭代的相对误差
    """
    ndim = tensor.dim()
    dims = tensor.shape

    factors = [torch.randn(d, rank) * 0.1 for d in dims]
    norm_X = tensor.norm()
    errors = []

    for iteration in range(max_iter):
        for mode in range(ndim):
            other_modes = [m for m in range(ndim) if m != mode]
            V = torch.ones(rank, rank)
            for m in other_modes:
                V = V * (factors[m].t() @ factors[m])

            X_unfolded = unfold_tensor(tensor, mode)
            kr_parts = [factors[m] for m in other_modes]
            kr = kr_parts[0]
            for m_idx in range(1, len(kr_parts)):
                kr = khatri_rao(kr, kr_parts[m_idx])
            factors[mode] = X_unfolded @ kr @ torch.linalg.pinv(V)

        recon = cp_reconstruct(factors, torch.ones(rank))
        error = (tensor - recon).norm() / norm_X
        errors.append(error.item())

        if verbose and iteration % 20 == 0:
            print(f"  Iteration {iteration}: relative error = {error.item():.6f}")

        if iteration > 0 and abs(errors[-2] - errors[-1]) < tol:
            if verbose:
                print(f"  Converged at iteration {iteration}")
            break

    for mode in range(ndim):
        col_norms = torch.norm(factors[mode], dim=0).clamp(min=1e-10)
        factors[mode] = factors[mode] / col_norms
        if mode == 0:
            lambdas = col_norms
        else:
            lambdas = lambdas * col_norms

    return factors, lambdas, errors


def cp_reconstruct(factors: List[torch.Tensor], lambdas: torch.Tensor) -> torch.Tensor:
    """从CP分解结果重建原始张量

    Args:
        factors: 各模式因子矩阵列表
        lambdas: 权重向量

    Returns:
        重建张量
    """
    R = len(lambdas)
    result = torch.zeros(*[f.shape[0] for f in factors])
    for r in range(R):
        rank1_term = lambdas[r].clone()
        for mode, factor in enumerate(factors):
            shape = [1] * len(factors)
            shape[mode] = factor.shape[0]
            rank1_term = rank1_term * factor[:, r].reshape(shape)
        result = result + rank1_term
    return result


def compute_corcondia(tensor: torch.Tensor, factors: List[torch.Tensor],
                      lambdas: torch.Tensor) -> float:
    """计算核心一致性（CORCONDIA）诊断CP秩是否合适

    值接近100%表示秩合适，低于50%表示秩过大。

    Args:
        tensor: 原始张量
        factors: CP分解的因子矩阵
        lambdas: CP分解的权重

    Returns:
        核心一致性百分比 [0, 100]
    """
    R = len(lambdas)
    ndim = tensor.dim()

    core = tensor.clone()
    for mode in range(ndim):
        core = mode_n_product(core, torch.linalg.pinv(factors[mode]), mode)

    mask = torch.zeros_like(core)
    for r in range(R):
        idx = tuple([r] * ndim)
        mask[idx] = 1.0

    core_diag = core * mask
    core_off = core * (1 - mask)

    numerator = (core_off ** 2).sum()
    denominator = (core_diag ** 2).sum() + numerator

    if denominator < 1e-10:
        return 100.0

    corcondia = 100.0 * (1.0 - numerator / denominator)
    return max(0.0, min(100.0, corcondia.item()))


print("=== CP分解基础验证 ===")

I, J, K = 64, 128, 32
R_true = 8
A_true = torch.randn(I, R_true)
B_true = torch.randn(J, R_true)
C_true = torch.randn(K, R_true)
lambdas_true = torch.abs(torch.randn(R_true)) + 1.0
X_lowrank = cp_reconstruct([A_true, B_true, C_true], lambdas_true)

print(f"原始3D张量: [{I}, {J}, {K}], 参数量: {X_lowrank.numel():,}")
print(f"真实CP秩: {R_true}")
print(f"CP分解参数量(R={R_true}): {R_true * (I + J + K):,}")

print(f"\n--- CP分解不同秩的压缩效果 ---")
for R in [4, 8, 16, 32]:
    factors, lambdas, errors = cp_als(X_lowrank, rank=R, max_iter=200)
    X_recon = cp_reconstruct(factors, lambdas)
    mse = F.mse_loss(X_lowrank, X_recon)
    rel_error = errors[-1]
    cp_params = R * (I + J + K)
    corcondia = compute_corcondia(X_lowrank, factors, lambdas)
    print(f"  R={R:<4} 参数量={cp_params:,}({cp_params/X_lowrank.numel():.2%}), "
          f"MSE={mse:.6f}, 相对误差={rel_error:.6f}, CORCONDIA={corcondia:.1f}%")

print(f"\n--- CP秩选择：拟合误差法（肘部法则） ---")
ranks_to_test = list(range(1, 21))
rel_errors = []
for R in ranks_to_test:
    factors, lambdas, errors = cp_als(X_lowrank, rank=R, max_iter=200)
    rel_errors.append(errors[-1])

print(f"  {'Rank':<6} {'相对误差':<15} {'误差下降':<15}")
for i, R in enumerate(ranks_to_test):
    delta = rel_errors[i-1] - rel_errors[i] if i > 0 else float('inf')
    marker = " <-- 拐点" if R == R_true else ""
    print(f"  {R:<6} {rel_errors[i]:<15.6f} {delta:<15.6f}{marker}")

print(f"\n--- CP vs Tucker 参数量对比 ---")
print(f"  张量形状: [{I}, {J}, {K}]")
for R in [8, 16, 32]:
    cp_params = R * (I + J + K)
    tucker_ranks = [min(R, I), min(R, J), min(R, K)]
    tucker_params = 1
    for r in tucker_ranks:
        tucker_params *= r
    tucker_params += sum(d * r for d, r in zip([I, J, K], tucker_ranks))
    print(f"  R={R}: CP参数={cp_params:,}, Tucker参数={tucker_params:,}, "
          f"CP更少{tucker_params/cp_params:.1f}倍")

---
### CP分解压缩卷积层

#### 卷积核的CP分解

对于4D卷积核$W \in \mathbb{R}^{C_{out} \times C_{in} \times K_H \times K_W}$，CP分解将其展开为3D张量后分解：

1. **空间维度合并**：将$K_H \times K_W$合并为一个维度，得到$W' \in \mathbb{R}^{C_{out} \times C_{in} \times (K_H \cdot K_W)}$
2. **CP-3分解**：$W' \approx \sum_{r=1}^{R} \lambda_r a_r \circ b_r \circ c_r$
   - $a_r \in \mathbb{R}^{C_{out}}$：输出通道因子
   - $b_r \in \mathbb{R}^{C_{in}}$：输入通道因子
   - $c_r \in \mathbb{R}^{K_H \cdot K_W}$：空间因子（可reshape回$K_H \times K_W$）

#### 等价卷积级联

CP分解后的卷积等价于以下级联操作：
1. **1×1卷积**（输入通道压缩）：$C_{in} \to R$，权重为$B^T \cdot \text{diag}(\lambda)$
2. **分组深度卷积**（R组深度卷积）：每组1个通道，核大小$K_H \times K_W$，权重为reshape后的$C$
3. **1×1卷积**（输出通道恢复）：$R \to C_{out}$，权重为$A$

#### 计算量分析

| 运算 | 原始卷积 | CP分解后 |
|------|---------|-------------|
| FLOPs | $C_{out} \cdot C_{in} \cdot K^2 \cdot H_{out} \cdot W_{out}$ | $(C_{in} \cdot R + R \cdot K^2 + R \cdot C_{out}) \cdot H_{out} \cdot W_{out}$ |
| 参数量 | $C_{out} \cdot C_{in} \cdot K^2$ | $R \cdot (C_{out} + C_{in} + K^2)$ |

与Tucker-2分解对比：
- Tucker-2参数量：$C_{in} \cdot R_{in} + R_{in} \cdot R_{out} \cdot K^2 + R_{out} \cdot C_{out}$
- CP参数量：$R \cdot (C_{out} + C_{in} + K^2)$
- 当$R \approx R_{in} \approx R_{out}$时，CP参数量通常更少（没有$R_{in} \cdot R_{out} \cdot K^2$项），但表达能力也较弱

In [ ]:
class CPCompressedConv2d(nn.Module):
    """CP分解压缩的Conv2d层

    将原始卷积分解为三个级联操作：
    1. 1x1卷积: C_in -> R (输入通道压缩)
    2. 深度卷积: R组, kernel_size=原始kernel_size (空间卷积)
    3. 1x1卷积: R -> C_out (输出通道恢复)

    CP分解相比Tucker分解参数更少，但表达能力较弱。
    适合对参数量要求极致的场景。
    """

    def __init__(
        self,
        conv1x1_in: nn.Conv2d,
        conv_depthwise: nn.Conv2d,
        conv1x1_out: nn.Conv2d,
        original_in_channels: int,
        original_out_channels: int,
    ):
        super().__init__()
        self.conv1x1_in = conv1x1_in
        self.conv_depthwise = conv_depthwise
        self.conv1x1_out = conv1x1_out
        self.original_in_channels = original_in_channels
        self.original_out_channels = original_out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1x1_in(x)
        x = self.conv_depthwise(x)
        x = self.conv1x1_out(x)
        return x

    @classmethod
    def from_conv2d(
        cls,
        conv: nn.Conv2d,
        rank: int,
        max_als_iter: int = 200,
    ) -> 'CPCompressedConv2d':
        """从标准Conv2d层创建CP压缩版本

        Args:
            conv: 原始nn.Conv2d层
            rank: CP秩
            max_als_iter: ALS最大迭代次数
        """
        W = conv.weight.data
        C_out, C_in, K_H, K_W = W.shape
        stride = conv.stride
        padding = conv.padding

        W_3d = W.reshape(C_out, C_in, K_H * K_W)
        factors, lambdas, _ = cp_als(W_3d, rank=rank, max_iter=max_als_iter)

        A = factors[0]
        B = factors[1]
        C_spatial = factors[2]

        conv1x1_in = nn.Conv2d(
            C_in, rank, kernel_size=1, bias=False,
            stride=1, padding=0
        )
        conv1x1_in.weight.data = (B * lambdas.unsqueeze(0)).t().unsqueeze(-1).unsqueeze(-1)

        conv_depthwise = nn.Conv2d(
            rank, rank, kernel_size=(K_H, K_W),
            stride=stride, padding=padding, bias=False,
            groups=rank
        )
        spatial_weights = C_spatial.t().reshape(rank, 1, K_H, K_W)
        conv_depthwise.weight.data = spatial_weights

        conv1x1_out = nn.Conv2d(
            rank, C_out, kernel_size=1, bias=True,
            stride=1, padding=0
        )
        conv1x1_out.weight.data = A.unsqueeze(-1).unsqueeze(-1)
        if conv.bias is not None:
            conv1x1_out.bias.data = conv.bias.data.clone()
        else:
            conv1x1_out.bias.data.zero_()

        return cls(conv1x1_in, conv_depthwise, conv1x1_out, C_in, C_out)

    def extra_repr(self) -> str:
        r = self.conv1x1_in.out_channels
        return (f'original: ({self.original_in_channels} -> {self.original_out_channels}), '
                f'compressed: ({self.original_in_channels} -> {r} -> {r} -> {self.original_out_channels})')


def compute_cp_conv_flops(cp_conv: CPCompressedConv2d, input_h: int, input_w: int) -> int:
    """计算CP压缩Conv2d的FLOPs"""
    flops = 0
    h, w = input_h, input_w
    for conv in [cp_conv.conv1x1_in, cp_conv.conv_depthwise, cp_conv.conv1x1_out]:
        C_out, C_in_k, K_H, K_W = conv.weight.shape
        out_h = (h + 2 * conv.padding[0] - K_H) // conv.stride[0] + 1
        out_w = (w + 2 * conv.padding[1] - K_W) // conv.stride[1] + 1
        flops += 2 * C_out * C_in_k * K_H * K_W * out_h * out_w
        h, w = out_h, out_w
    return flops


C_out_cp, C_in_cp, K_cp = 128, 256, 3
conv_orig_cp = nn.Conv2d(C_in_cp, C_out_cp, kernel_size=K_cp, padding=1, bias=True)

print(f"=== CP分解压缩Conv2d ===")
print(f"原始卷积: Conv2d({C_in_cp}, {C_out_cp}, kernel_size={K_cp})")
print(f"原始参数量: {sum(p.numel() for p in conv_orig_cp.parameters()):,}")

input_h_cp, input_w_cp = 32, 32
orig_flops_cp = compute_conv_flops(conv_orig_cp, input_h_cp, input_w_cp)
print(f"原始FLOPs: {orig_flops_cp:,}")

x_conv_cp = torch.randn(1, C_in_cp, input_h_cp, input_w_cp)
with torch.no_grad():
    out_orig_cp = conv_orig_cp(x_conv_cp)

print(f"\n--- CP分解 vs Tucker分解参数量对比 ---")
for R in [32, 64, 96, 128]:
    cp_params = R * (C_out_cp + C_in_cp + K_cp * K_cp)
    tucker_params = C_in_cp * R + R * R * K_cp * K_cp + R * C_out_cp
    print(f"  R={R:<4}: CP参数={cp_params:,}, Tucker参数={tucker_params:,}, "
          f"CP更少{tucker_params/cp_params:.1f}倍")

print(f"\n--- CP分解不同秩的压缩效果 ---")
for R in [16, 32, 64, 96]:
    cp_conv = CPCompressedConv2d.from_conv2d(conv_orig_cp, rank=R)
    cp_params = sum(p.numel() for p in cp_conv.parameters())
    cp_flops = compute_cp_conv_flops(cp_conv, input_h_cp, input_w_cp)
    with torch.no_grad():
        out_cp = cp_conv(x_conv_cp)
    cp_diff = (out_orig_cp - out_cp).norm() / out_orig_cp.norm() * 100

    print(f"  R={R:<4}: 参数={cp_params:,}({cp_params/conv_orig_cp.weight.numel():.1%}), "
          f"FLOPs={cp_flops:,}({cp_flops/orig_flops_cp:.1%}), 输出差异={cp_diff:.2f}%")

---
## 1.4.4 低秩重参数化（LoRA-style Factorization）

### 什么是LoRA式低秩分解？

冻结原始权重W，添加低秩增量 ΔW = BA，其中$A \in \mathbb{R}^{r \times n}$, $B \in \mathbb{R}^{m \times r}$。推理时可将BA合并回W，无额外推理开销。虽主要用于微调，但其低秩思想可用于压缩。

### 为什么LoRA式分解有效？

1. **零推理开销**：合并后$W' = W + \frac{\alpha}{r}BA$，与原始线性层完全等价，无额外计算。
2. **训练高效**：仅训练$r(m+n)$个参数（通常<1%总参数），显存需求大幅降低。
3. **多任务切换**：同一基座模型+不同LoRA适配器，可按需切换任务，无需加载多个完整模型。

### LoRA的数学公式

$$h = xW^T + x \cdot \frac{\alpha}{r} \cdot A^T B^T = x(W + \frac{\alpha}{r}BA)^T$$

**合并后**：
$$W' = W + \frac{\alpha}{r}BA$$

其中：
- $W \in \mathbb{R}^{m \times n}$：原始冻结权重
- $A \in \mathbb{R}^{r \times n}$：LoRA下投影矩阵，可训练，Kaiming初始化
- $B \in \mathbb{R}^{m \times r}$：LoRA上投影矩阵，可训练，零初始化
- $r$：LoRA秩，远小于$m$和$n$
- $\alpha$：缩放因子，控制LoRA增量的影响强度
- $\frac{\alpha}{r}$：缩放系数，确保不同rank下适配器的输出量级一致
- $B$零初始化确保训练开始时$\Delta W = 0$，不改变原始模型输出

### LoRA参数选择指南

#### 如何选择LoRA秩$r$？

LoRA秩$r$决定了低秩增量的表达能力，选择依据任务复杂度：

| 任务类型 | 推荐rank | 说明 |
|---------|---------|------|
| 简单任务（风格迁移、小数据微调） | 4-8 | 低秩足以捕获风格差异 |
| 中等任务（领域适配、指令微调） | 8-32 | 需要更多自由度适配新领域 |
| 复杂任务（新语言学习、多任务） | 32-64 | 需要高秩捕获复杂模式 |

实践经验：
- 从较小的rank（如8）开始，逐步增大直到验证集性能不再提升
- rank加倍通常比微调更多epoch更有效
- 不同层可以使用不同rank（如注意力层rank=8，FFN层rank=16）

#### 如何选择缩放因子$\alpha$？

$\alpha$控制LoRA增量的影响强度，有效缩放为$\alpha/r$：

- **常用设置**：$\alpha = 2r$（即$\alpha/r = 2$），这是最稳定的默认选择
- **$\alpha/r > 1$**：LoRA增量影响更大，适合任务与预训练差异大的场景
- **$\alpha/r < 1$**：LoRA增量影响更小，适合任务与预训练接近的场景
- **$\alpha/r = 1$**：即$\alpha = r$，无额外缩放，某些实现中使用

#### 如何选择目标模块？

并非所有层都需要加LoRA，选择策略：
- **注意力层**：Q、V投影最有效（捕获注意力模式变化），K投影效果较小
- **FFN层**：up和gate投影有效（捕获知识变化）
- **通常跳过**：embedding层、输出层（LayerNorm也不加）
- **经验法则**：先只对Q、V加LoRA，效果不够再加FFN层

In [ ]:
class LoRALinear(nn.Module):
    """LoRA线性层：原始权重冻结 + 低秩适配

    实际应用中的关键设计：
    - A使用Kaiming初始化，B零初始化，确保训练开始时ΔW=0
    - scaling = alpha/rank，确保不同rank下适配器输出量级一致
    - 合并后推理零开销，与原始线性层完全等价
    - 支持多个LoRA适配器切换（多任务场景）
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        rank: int = 8,
        alpha: float = 16.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.merged = False

        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)

        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        self.dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

    @classmethod
    def from_linear(cls, linear: nn.Linear, rank: int = 8, alpha: float = 16.0,
                    dropout: float = 0.0) -> 'LoRALinear':
        """从已有nn.Linear层创建LoRA版本，保留原始权重"""
        lora_layer = cls(
            in_features=linear.in_features,
            out_features=linear.out_features,
            rank=rank,
            alpha=alpha,
            dropout=dropout,
        )
        lora_layer.weight.data.copy_(linear.weight.data)
        if linear.bias is not None:
            lora_layer.bias.data.copy_(linear.bias.data)
        return lora_layer

    def merge_weights(self) -> None:
        """将LoRA权重合并到主权重，推理时无额外开销"""
        if not self.merged:
            self.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True
            self.lora_A.requires_grad_(False)
            self.lora_B.requires_grad_(False)

    def unmerge_weights(self) -> None:
        """取消合并（用于切换LoRA适配器）"""
        if self.merged:
            self.weight.data -= self.scaling * (self.lora_B @ self.lora_A)
            self.merged = False
            self.lora_A.requires_grad_(True)
            self.lora_B.requires_grad_(True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.merged:
            return F.linear(x, self.weight, self.bias)
        result = F.linear(x, self.weight, self.bias)
        lora_out = self.dropout(x) @ self.lora_A.t() @ self.lora_B.t()
        result += lora_out * self.scaling
        return result

    def extra_repr(self) -> str:
        return (f'in_features={self.in_features}, out_features={self.out_features}, '
                f'rank={self.rank}, alpha={self.alpha}, merged={self.merged}')


in_f, out_f = 512, 256
lora_layer = LoRALinear(in_f, out_f, rank=8, alpha=16)

x = torch.randn(4, in_f)

with torch.no_grad():
    out_unmerged = lora_layer(x)

lora_layer.merge_weights()
with torch.no_grad():
    out_merged = lora_layer(x)

merge_diff = (out_unmerged - out_merged).norm() / out_unmerged.norm() * 100

base_params = in_f * out_f
lora_params = in_f * 8 + out_f * 8

print(f"=== LoRA低秩重参数化 ===")
print(f"原始权重参数: {base_params:,}")
print(f"LoRA参数(rank=8): {lora_params:,} ({lora_params/base_params:.2%})")
print(f"合并前后输出差异: {merge_diff:.6f}%")
print(f"\n关键: 合并后推理无额外开销，与原始线性层完全等价")

print(f"\n--- 不同rank的参数量对比 ---")
for r in [1, 2, 4, 8, 16, 32, 64]:
    lora_p = in_f * r + out_f * r
    print(f"  rank={r:<3} LoRA参数={lora_p:<8} 占原始{base_params}的{lora_p/base_params:.2%}")

### LoRA微调效果验证

#### 验证什么？

验证LoRA微调的实际效果：仅训练少量LoRA参数，观察模型性能变化，以及合并后推理是否零开销。

#### 验证要点

1. **参数量对比**：LoRA参数量 vs 原始模型参数量，通常<1%
2. **性能对比**：LoRA微调前后的loss/精度变化
3. **合并验证**：合并$W' = W + \frac{\alpha}{r}BA$后，输出与未合并是否一致
4. **推理速度**：合并后的推理速度应与原始模型完全相同

In [ ]:
def apply_lora_to_model(
    model: nn.Module,
    rank: int = 8,
    alpha: float = 16.0,
    target_modules: Optional[List[str]] = None,
) -> nn.Module:
    """将模型中的Linear层替换为LoRA版本

    Args:
        model: 原始模型
        rank: LoRA秩
        alpha: LoRA缩放因子
        target_modules: 目标模块名子串列表，None表示所有Linear层

    Returns:
        修改后的模型（原地修改）
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if target_modules and not any(t in name for t in target_modules):
                continue
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            parent = model
            for part in parent_name.split('.'):
                if part:
                    parent = getattr(parent, part)
            lora_layer = LoRALinear.from_linear(module, rank=rank, alpha=alpha)
            setattr(parent, child_name, lora_layer)
    return model


def get_lora_params(model: nn.Module) -> List[nn.Parameter]:
    """获取模型中所有LoRA可训练参数"""
    return [p for n, p in model.named_parameters() if 'lora_' in n and p.requires_grad]


def freeze_non_lora_params(model: nn.Module) -> None:
    """冻结所有非LoRA参数"""
    for n, p in model.named_parameters():
        if 'lora_' not in n:
            p.requires_grad = False


def merge_lora_weights(model: nn.Module) -> None:
    """合并模型中所有LoRA层的权重"""
    for module in model.modules():
        if isinstance(module, LoRALinear) and not module.merged:
            module.merge_weights()


class LoRAModel(nn.Module):
    def __init__(self, dim: int = 64, n_classes: int = 10, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.layer1 = LoRALinear(dim, dim * 2, rank=rank, alpha=alpha)
        self.layer2 = LoRALinear(dim * 2, dim, rank=rank, alpha=alpha)
        self.layer3 = LoRALinear(dim, n_classes, rank=rank, alpha=alpha)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)


dim, n_classes = 64, 10
x_data = torch.randn(256, dim, device=device)
y_data = torch.randint(0, n_classes, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

model_lora = LoRAModel(dim=dim, n_classes=n_classes, rank=8, alpha=16).to(device)
freeze_non_lora_params(model_lora)

lora_params = sum(p.numel() for p in get_lora_params(model_lora))
total_params = sum(p.numel() for p in model_lora.parameters())

optimizer = torch.optim.AdamW(get_lora_params(model_lora), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

losses = []
for epoch in range(30):
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model_lora(x)
        loss = F.cross_entropy(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

with torch.no_grad():
    acc_lora = (model_lora(x_data).argmax(1).cpu() == y_data).float().mean()

x_check = torch.randn(4, dim, device=device)
with torch.no_grad():
    out_before_merge = model_lora(x_check).clone()

merge_lora_weights(model_lora)

with torch.no_grad():
    out_after_merge = model_lora(x_check).clone()
    acc_merged = (model_lora(x_data).argmax(1).cpu() == y_data).float().mean()

merge_diff = (out_before_merge - out_after_merge).norm() / out_before_merge.norm() * 100

print(f"=== LoRA微调效果 ===")
print(f"总参数量: {total_params:,}")
print(f"LoRA可训练参数: {lora_params:,} ({lora_params/total_params:.2%})")
print(f"冻结参数: {total_params - lora_params:,}")
print(f"微调后准确率: {acc_lora:.4f}")
print(f"合并后准确率: {acc_merged:.4f}")
print(f"合并前后输出差异: {merge_diff:.6f}%")
print(f"训练损失: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"\nLoRA核心优势: 仅训练极少参数，合并后推理零开销")

---
## 1.4.5 低秩压缩后的微调恢复

### 为什么需要微调恢复？

SVD和Tucker分解是纯数学分解，截断小奇异值会引入近似误差。对于精度敏感的应用，压缩后通常需要少量微调来恢复精度：

1. **误差累积**：多层压缩的误差会逐层传播放大
2. **非最优分解**：SVD/Tucker是逐层独立分解，未考虑层间交互
3. **任务适配**：压缩后的权重可能偏离特定任务的最优解

### 微调策略

- **全参数微调**：解冻所有参数训练，效果最好但成本最高
- **LoRA微调**：在压缩后的模型上再加LoRA，仅训练低秩增量
- **学习率策略**：压缩模型微调通常使用较小学习率（原始的1/10~1/5）
- **渐进式解冻**：先训练最后几层，再逐步解冻前面的层

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, dim: int = 64, n_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim * 2),
            nn.GELU(),
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Linear(dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_model(model: nn.Module, loader: torch.utils.data.DataLoader,
                n_epochs: int = 20, lr: float = 1e-3, device: torch.device = device) -> List[float]:
    """训练模型并返回每个epoch的loss"""
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        n_batches = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = F.cross_entropy(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        losses.append(epoch_loss / n_batches)
    return losses


def evaluate_model(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> float:
    """评估模型准确率"""
    model.eval()
    with torch.no_grad():
        acc = (model(x.to(device)).argmax(1).cpu() == y).float().mean().item()
    model.train()
    return acc


dim, n_classes = 64, 10
n_samples = 1024
x_data = torch.randn(n_samples, dim)
y_data = torch.randint(0, n_classes, (n_samples,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)
x_eval = torch.randn(256, dim)
y_eval = torch.randint(0, n_classes, (256,))

print("=== Step 1: 训练原始模型 ===")
original_model = SimpleClassifier(dim=dim, n_classes=n_classes)
train_model(original_model, loader, n_epochs=30, lr=1e-3)
acc_orig = evaluate_model(original_model, x_eval, y_eval)
orig_params = sum(p.numel() for p in original_model.parameters())
print(f"原始模型准确率: {acc_orig:.4f}, 参数量: {orig_params:,}")

print(f"\n=== Step 2: SVD压缩（无微调） ===")
for rank_ratio in [0.25, 0.5, 0.75]:
    compressed = copy.deepcopy(original_model)
    replace_linear_with_svd(compressed, rank_ratio=rank_ratio, min_rank=4)
    acc_svd = evaluate_model(compressed, x_eval, y_eval)
    comp_params = sum(p.numel() for p in compressed.parameters())
    print(f"  rank比例={rank_ratio:.0%}: 准确率={acc_svd:.4f}({acc_svd-acc_orig:+.4f}), "
          f"参数量={comp_params:,}({comp_params/orig_params:.1%})")

print(f"\n=== Step 3: SVD压缩 + 微调恢复 ===")
for rank_ratio in [0.25, 0.5, 0.75]:
    compressed = copy.deepcopy(original_model)
    replace_linear_with_svd(compressed, rank_ratio=rank_ratio, min_rank=4)
    acc_before_ft = evaluate_model(compressed, x_eval, y_eval)
    train_model(compressed, loader, n_epochs=15, lr=1e-4)
    acc_after_ft = evaluate_model(compressed, x_eval, y_eval)
    comp_params = sum(p.numel() for p in compressed.parameters())
    print(f"  rank比例={rank_ratio:.0%}: 压缩后={acc_before_ft:.4f}, 微调后={acc_after_ft:.4f}({acc_after_ft-acc_orig:+.4f}), "
          f"参数量={comp_params:,}({comp_params/orig_params:.1%})")

print(f"\n=== Step 4: SVD压缩 + LoRA微调恢复 ===")
for rank_ratio in [0.25, 0.5, 0.75]:
    compressed = copy.deepcopy(original_model)
    replace_linear_with_svd(compressed, rank_ratio=rank_ratio, min_rank=4)

    for name, module in compressed.named_modules():
        if isinstance(module, SVDCompressedLinear):
            lora_a = nn.Parameter(torch.randn(4, module.in_features) * 0.01)
            lora_b = nn.Parameter(torch.zeros(module.out_features, 4))
            module.lora_A = lora_a
            module.lora_B = lora_b
            module.lora_scaling = 8.0 / 4
            module._lora_forward = module.forward
            orig_forward = module.forward
            def make_forward(orig_fwd, a, b, scale):
                def new_forward(x):
                    result = orig_fwd(x)
                    result += (x @ a.t() @ b.t()) * scale
                    return result
                return new_forward
            module.forward = make_forward(orig_forward, lora_a, lora_b, module.lora_scaling)

    for p in compressed.parameters():
        p.requires_grad = False
    for name, p in compressed.named_parameters():
        if 'lora_' in name:
            p.requires_grad = True

    lora_only_params = sum(p.numel() for p in compressed.parameters() if p.requires_grad)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, compressed.parameters()), lr=1e-3)
    compressed.train()
    for epoch in range(15):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = compressed(x)
            loss = F.cross_entropy(out, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    acc_lora_ft = evaluate_model(compressed, x_eval, y_eval)
    comp_params = sum(p.numel() for p in compressed.parameters())
    print(f"  rank比例={rank_ratio:.0%}: LoRA微调后={acc_lora_ft:.4f}({acc_lora_ft-acc_orig:+.4f}), "
          f"LoRA参数={lora_only_params:,}, 总参数={comp_params:,}({comp_params/orig_params:.1%})")

---
## 1.4.6 低秩分解方法综合对比与实战Pipeline

### 不同低秩分解方法对比

| 方法 | 原理 | 适用对象 | 推理开销 | 是否需微调 | 典型场景 |
|------|------|---------|---------|-----------|----------|
| 截断SVD | 保留前r个奇异值 | 2D矩阵 | 两次小矩阵乘 | 可选 | 线性层压缩 |
| Tucker分解 | 多模式分解 | N维张量 | 核张量+因子矩阵 | 可选 | 卷积核压缩 |
| CP分解 | 秩1项求和 | N维张量 | 秩1项求和 | 可选 | 结构化压缩 |
| LoRA | 低秩增量$\Delta W=BA$ | 2D矩阵 | 合并后零开销 | 必须 | 微调+压缩 |

### 选择指南

- **仅需压缩（无训练数据）**：截断SVD，选择合适的rank保留能量
- **卷积模型压缩**：Tucker分解（灵活，各维度独立选秩）或CP分解（紧凑，参数更少）
- **极致参数压缩**：CP分解，参数量最少但表达能力较弱，适合对参数量要求严格的场景
- **微调+压缩**：LoRA，仅训练低秩参数，合并后零开销
- **联合优化**：量化 + 低秩分解，先SVD/Tucker/CP压缩再量化，效果叠加

### 产业级压缩Pipeline

```
1. 分析模型 → 确定压缩目标（参数量/FLOPs/延迟）
2. 敏感性分析 → 逐层评估压缩对精度的影响
3. 选择方法 → Linear层用SVD，Conv层用Tucker/CP，微调用LoRA
4. 确定秩 → SVD/Tucker用能量保留法，CP用拟合误差法+CORCONDIA，LoRA依据任务复杂度
5. 压缩 → 逐层分解替换
6. 微调恢复 → 小学习率全参数微调或LoRA微调
7. 验证 → 精度/延迟/内存全面评估
```

In [ ]:
def model_compression_pipeline(
    model: nn.Module,
    target_params_ratio: float = 0.5,
    x_sample: Optional[torch.Tensor] = None,
    skip_first_last: bool = True,
) -> Tuple[nn.Module, Dict]:
    """低秩压缩Pipeline：自动分析、压缩、验证

    Args:
        model: 原始模型
        target_params_ratio: 目标参数量比例
        x_sample: 用于验证的输入样本
        skip_first_last: 是否跳过首尾层

    Returns:
        compressed_model: 压缩后的模型
        report: 压缩报告
    """
    model = copy.deepcopy(model)
    original_params = sum(p.numel() for p in model.parameters())
    target_params = int(original_params * target_params_ratio)

    report = {
        'original_params': original_params,
        'target_params': target_params,
        'target_ratio': target_params_ratio,
        'compressed_layers': [],
    }

    if x_sample is not None:
        with torch.no_grad():
            out_orig = model(x_sample)
        report['output_shape'] = list(out_orig.shape)

    layer_info = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.dim() == 2:
            out_f, in_f = module.weight.shape
            layer_info.append({
                'name': name, 'type': 'linear',
                'out_f': out_f, 'in_f': in_f,
                'params': module.weight.numel() + (module.bias.numel() if module.bias is not None else 0),
            })
        elif isinstance(module, nn.Conv2d) and module.weight.dim() == 4:
            C_out, C_in, K_H, K_W = module.weight.shape
            layer_info.append({
                'name': name, 'type': 'conv2d',
                'C_out': C_out, 'C_in': C_in,
                'K_H': K_H, 'K_W': K_W,
                'params': sum(p.numel() for p in module.parameters()),
            })

    layer_info.sort(key=lambda x: x['params'], reverse=True)

    current_params = original_params
    for info in layer_info:
        if current_params <= target_params:
            break

        name = info['name']
        if skip_first_last and info == layer_info[0]:
            continue

        module = dict(model.named_modules())[name]

        if info['type'] == 'linear':
            out_f, in_f = info['out_f'], info['in_f']
            for rank_ratio in [0.75, 0.5, 0.25]:
                rank = max(4, int(min(out_f, in_f) * rank_ratio))
                new_params = out_f * rank + rank * in_f
                if current_params - info['params'] + new_params <= target_params:
                    compressed = SVDCompressedLinear.from_linear(module, rank=rank)
                    parent_name = '.'.join(name.split('.')[:-1])
                    child_name = name.split('.')[-1]
                    parent = model
                    for part in parent_name.split('.'):
                        if part:
                            parent = getattr(parent, part)
                    setattr(parent, child_name, compressed)
                    current_params = current_params - info['params'] + new_params
                    report['compressed_layers'].append({
                        'name': name, 'method': 'SVD', 'rank': rank,
                        'orig_params': info['params'], 'new_params': new_params,
                    })
                    break

        elif info['type'] == 'conv2d':
            C_out, C_in = info['C_out'], info['C_in']
            K_H, K_W = info['K_H'], info['K_W']
            for rank_ratio in [0.5, 0.25]:
                rank_in = max(4, int(C_in * rank_ratio))
                rank_out = max(4, int(C_out * rank_ratio))
                new_params = C_in * rank_in + rank_in * rank_out * K_H * K_W + rank_out * C_out
                if current_params - info['params'] + new_params <= target_params:
                    tucker_conv = TuckerCompressedConv2d.from_conv2d(module, rank_in=rank_in, rank_out=rank_out)
                    parent_name = '.'.join(name.split('.')[:-1])
                    child_name = name.split('.')[-1]
                    parent = model
                    for part in parent_name.split('.'):
                        if part:
                            parent = getattr(parent, part)
                    setattr(parent, child_name, tucker_conv)
                    current_params = current_params - info['params'] + new_params
                    report['compressed_layers'].append({
                        'name': name, 'method': 'Tucker',
                        'rank_in': rank_in, 'rank_out': rank_out,
                        'orig_params': info['params'], 'new_params': new_params,
                    })
                    break

    report['final_params'] = current_params
    report['compression_ratio'] = original_params / max(current_params, 1)
    report['param_reduction'] = 1 - current_params / original_params

    if x_sample is not None:
        with torch.no_grad():
            out_comp = model(x_sample)
        report['output_diff_pct'] = ((out_orig - out_comp).norm() / out_orig.norm() * 100).item()

    return model, report


print(f"{'方法':<25} {'适用场景':<25} {'推理开销':<15} {'需要微调':<10}")
print("-" * 75)
methods = [
    ("SVD截断", "权重矩阵压缩", "减少(两步矩阵乘)", "可选"),
    ("Tucker分解", "高维张量(如Conv)", "减少", "可选"),
    ("CP分解", "结构化张量压缩", "减少", "可选"),
    ("LoRA重参数化", "微调/增量压缩", "零(合并后)", "必须"),
]
for name, scene, overhead, finetune in methods:
    print(f"{name:<25} {scene:<25} {overhead:<15} {finetune:<10}")

print(f"\n=== 产业级压缩Pipeline演示 ===")
pipeline_model = SmallTransformer(dim=64, n_heads=4, n_classes=10).to(device)
x_pipe = torch.randn(4, 8, 64, device=device)

for target_ratio in [0.75, 0.5, 0.25]:
    compressed_model, report = model_compression_pipeline(
        pipeline_model, target_params_ratio=target_ratio, x_sample=x_pipe
    )
    print(f"\n--- 目标参数比例: {target_ratio:.0%} ---")
    print(f"  原始参数: {report['original_params']:,}")
    print(f"  压缩后参数: {report['final_params']:,} ({report['final_params']/report['original_params']:.1%})")
    print(f"  压缩比: {report['compression_ratio']:.2f}x")
    print(f"  输出差异: {report.get('output_diff_pct', 'N/A'):.2f}%" if 'output_diff_pct' in report else "")
    print(f"  压缩层数: {len(report['compressed_layers'])}")
    for layer in report['compressed_layers']:
        method = layer['method']
        if method == 'SVD':
            print(f"    {layer['name']}: SVD rank={layer['rank']}, "
                  f"{layer['orig_params']}->{layer['new_params']} params")
        elif method == 'Tucker':
            print(f"    {layer['name']}: Tucker ranks=({layer['rank_in']},{layer['rank_out']}), "
                  f"{layer['orig_params']}->{layer['new_params']} params")

---
## 总结

### 核心要点

1. **SVD分解**：最经典的低秩分解方法，适用于2D权重矩阵，Frobenius范数下的最优低秩近似，无需训练数据。秩选择推荐能量保留法（95%）
2. **Tucker分解**：SVD在高维张量上的推广，适用于4D卷积核压缩，允许各维度独立选择秩，可等价转换为级联卷积。Tucker-2分解仅压缩通道维度，空间维度保持不变
3. **CP分解**：将张量分解为秩1项之和，参数量比Tucker更少，条件唯一。适合极致压缩场景，但表达能力较弱。秩选择推荐拟合误差法（肘部法则）和CORCONDIA诊断
4. **LoRA**：低秩增量分解，冻结原始权重仅训练低秩部分，合并后推理零开销，适合微调场景。rank选择依据任务复杂度（4-64），alpha通常取2×rank
5. **微调恢复**：压缩后通常需要微调恢复精度，可用全参数微调或LoRA微调

### 实战建议

- **先分析后压缩**：先做敏感性分析，确定哪些层对压缩最敏感
- **选择性压缩**：优先压缩参数量大且冗余度高的层，保留敏感层
- **秩选择**：SVD/Tucker推荐能量保留法（95%-99%），CP推荐拟合误差法+CORCONDIA，LoRA依据任务复杂度
- **方法选择**：线性层用SVD，卷积层用Tucker（灵活）或CP（紧凑），微调用LoRA
- **组合优化**：低秩分解 + 量化可以叠加使用，先低秩压缩再量化
- **验证充分**：压缩后必须在真实数据上验证精度，不能只看参数量